# Preprocessing fitur IMF untuk regresi AHI

Notebook ini membaca tiga sensor, memvalidasi data, membentuk split kelompok pasien, dan menyimpan feature image beserta metadata. Tidak ada training model.

Status sementara: provenance, salinan fitur antar-ID, perbedaan target pada salinan, dan AHI ekstrem masih menunggu konfirmasi klien. Semua target dan identitas dipertahankan. ID yang terhubung oleh fitur identik tiga sensor ditempatkan pada subset yang sama.

Tahapan: setup; audit/loading; complete-case; grouping; split; EDA; scaling training; encoding citra/mask; penyimpanan; validasi dan kesimpulan. Jalankan berurutan dari kernel baru. Output berisi data penelitian disimpan lokal dalam artifacts yang diabaikan Git.

In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'config.py').is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ['MPLBACKEND'] = 'Agg'

import json
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from src.audit import file_hash, run_audit
from src.config import AUDIT_DIR, ARTIFACTS_DIR, DATA_DIR, FEATURE_COLUMNS, FIGURES_DIR, POSITIONS, PREPROCESSING, PROCESSED_DIR, SENSOR_FILES
from src.preprocessing import assign_patient_components, build_sample_manifest, fit_scalers, load_sensor_tables, save_json, save_processed_dataset, split_manifest, validate_manifest

np.random.seed(PREPROCESSING.seed)
figure_dir = FIGURES_DIR / 'preprocessing'
figure_dir.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
colors = {'Flow': '#2166ac', 'PulseRate': '#b35806', 'Pleth': '#5e7d46'}

def show_figure(figure, name):
    figure.tight_layout()
    path = figure_dir / (name + '.png')
    figure.savefig(path, dpi=PREPROCESSING.figure_dpi, bbox_inches='tight', facecolor='white')
    plt.close(figure)
    display(Image(filename=str(path), width=1050))

print('Python:', sys.version.split()[0])
print('Status: sementara, tanpa training; seed:', PREPROCESSING.seed)
print('Ukuran citra:', PREPROCESSING.image_size)

## 1. Audit dan pembacaan sumber

Audit dijalankan kembali pada workbook aktual. Header dan whitespace metadata dinormalisasi di memori; ID tetap string. Schema, posisi, numerik finite, kunci IMF, dan konsistensi AHI divalidasi. Ketidakkonsistenan target menghentikan proses, tanpa averaging. Statistik audit bukan parameter scaler.

In [ ]:
run_audit()
tables, sources = load_sensor_tables()
audit_report = json.loads((AUDIT_DIR / 'dataset_audit.json').read_text(encoding='utf-8'))
assert not audit_report['blocking_issues']
display(pd.DataFrame([{'sensor': sensor, 'baris_IMF': len(frame), 'pasien': frame.id.nunique(), 'fitur_numerik': len(FEATURE_COLUMNS), 'ID_string': str(frame.id.dtype)} for sensor, frame in tables.items()]))

## 2. Complete-case dan pemeriksaan salinan

Satu sampel adalah pasangan pasien-posisi dengan satu AHI. IMF bukan sampel independen. Tabel target dibentuk setelah pengelompokan, sehingga tidak terjadi Cartesian product baris IMF.

Eksperimen utama memakai sampel dengan ketiga sensor. Fingerprint menggunakan indeks IMF numerik dan 11 fitur mentah seluruh sensor pada posisi yang sama. ID dan AHI tidak digunakan untuk menentukan kesamaan fitur. Salinan tidak dihapus; target berbeda antar-ID tetap dipertahankan.

In [ ]:
samples, excluded = build_sample_manifest(tables)
samples = assign_patient_components(samples)
repeated = samples[samples.duplicated('feature_fingerprint', keep=False)]
repeated_groups = repeated.groupby('feature_fingerprint')
display(pd.DataFrame([{'complete_case': len(samples), 'pasien': samples.id.nunique(), 'komponen_pasien': samples.patient_component.nunique(), 'pasangan_tidak_lengkap': len(excluded), 'sampel_fitur_identik': len(repeated), 'kelompok_fitur_identik': repeated_groups.ngroups, 'kelompok_identik_target_berbeda': int((repeated_groups.AHI.nunique() > 1).sum())}]))
if not excluded.empty:
    display(excluded.groupby(['Posisi', 'missing_sensors']).size().rename('sampel_dikeluarkan').reset_index())
fig, ax = plt.subplots(figsize=(7, 4))
counts = pd.DataFrame({sensor: frame.groupby('Posisi').id.nunique().reindex(POSITIONS) for sensor, frame in tables.items()})
counts.plot.bar(ax=ax, color=[colors[sensor] for sensor in SENSOR_FILES], rot=0)
ax.set(xlabel='Posisi tidur', ylabel='Pasangan pasien-posisi', title='Ketersediaan sensor sebelum complete-case')
show_figure(fig, 'sensor_availability')

## 3. Split kelompok pasien dan komponen salinan

GroupShuffleSplit memakai seed 42 dan rasio awal 70/15/15 pada kelompok. Semua posisi seorang pasien mengikuti kelompok yang sama. Komponen terhubung menjaga salinan yang melibatkan beberapa ID tetap dalam satu subset, termasuk hubungan transitif lintas posisi.

Rasio sampel/pasien dapat berbeda dari rasio kelompok. Split tidak diulang untuk mencari distribusi AHI atau hasil yang lebih baik. Statistik test hanya dilaporkan; tidak dipakai memilih preprocessing, seed, atau model.

In [ ]:
manifest = split_manifest(samples)
validate_manifest(manifest)
from src.preprocessing import split_statistics
statistics = split_statistics(manifest)
display(statistics.round(4))
overall = statistics[statistics.position == 'Overall'].copy()
overall['persen_sampel'] = 100 * overall.samples / len(manifest)
overall['persen_pasien'] = 100 * overall.patients / manifest.id.nunique()
display(overall[['split', 'samples', 'patients', 'components', 'persen_sampel', 'persen_pasien']].round(2))
fig, ax = plt.subplots(figsize=(7, 4))
manifest.groupby(['split', 'Posisi']).size().unstack().reindex(['train', 'validation', 'test'])[list(POSITIONS)].plot.bar(ax=ax, rot=0, color=['#2166ac', '#b35806', '#5e7d46'])
ax.set(xlabel='Subset', ylabel='Jumlah sampel', title='Distribusi posisi setelah split kelompok')
show_figure(fig, 'split_position_counts')

## 4. EDA AHI dan jumlah IMF

AHI dihitung sekali per pasien-posisi, bukan per IMF. Nilai nol dan ekstrem tetap ditampilkan. Penanda audit di atas 100 hanya menandai pemeriksaan sumber, bukan diagnosis atau aturan penghapusan. Test tidak digunakan untuk keputusan metode.

Distribusi jumlah IMF dan fitur berikut menggunakan baris training yang benar-benar memiliki tiga sensor. Padding tidak ikut membentuk statistik.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(manifest.AHI, bins=45, color='#2166ac', edgecolor='white')
axes[0].axvline(PREPROCESSING.ahi_audit_threshold, color='#b35806', linestyle=':', label='Penanda audit')
axes[0].set(xlabel='AHI asli', ylabel='Jumlah sampel', title='Distribusi AHI complete-case')
axes[0].legend()
axes[1].boxplot([manifest.loc[manifest.Posisi == position, 'AHI'] for position in POSITIONS], tick_labels=POSITIONS, showfliers=True)
axes[1].set(xlabel='Posisi tidur', ylabel='AHI asli', title='AHI menurut posisi, termasuk nilai ekstrem')
show_figure(fig, 'ahi_distribution')
display(pd.DataFrame([{'minimum_AHI': manifest.AHI.min(), 'median_AHI': manifest.AHI.median(), 'maksimum_AHI': manifest.AHI.max(), 'jumlah_AHI_nol': int((manifest.AHI == 0).sum()), 'di_atas_penanda_audit': int((manifest.AHI > PREPROCESSING.ahi_audit_threshold).sum())}]))
train_keys = pd.MultiIndex.from_frame(manifest.loc[manifest.split == 'train', ['id', 'Posisi']])
train_tables = {sensor: frame.loc[pd.MultiIndex.from_frame(frame[['id', 'Posisi']]).isin(train_keys)] for sensor, frame in tables.items()}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True)
for ax, (sensor, frame) in zip(axes, train_tables.items()):
    counts = frame.groupby(['id', 'Posisi']).size()
    ax.hist(counts, bins=np.arange(0.5, PREPROCESSING.max_imfs + 1.5), color=colors[sensor])
    ax.set(title=sensor, xlabel='Jumlah IMF tersedia', ylabel='Sampel training')
show_figure(fig, 'imf_counts_training')

## 5. Distribusi fitur dan korelasi training

Distribusi ditampilkan sebagai sign(x) log(1 + |x|) agar rentang lebar tetap terbaca. Transformasi ini hanya untuk grafik, bukan input model. Korelasi Pearson dihitung pada fitur mentah training; setiap baris merupakan IMF dan baris dalam pasien tidak dianggap independen untuk inferensi statistik.

Scaling model tetap menggunakan persentil 1 dan 99 pada fitur mentah training, sesuai konfigurasi yang ditentukan sebelum melihat test.

In [ ]:
for sensor, frame in train_tables.items():
    fig, axes = plt.subplots(3, 4, figsize=(14, 9))
    for ax, feature in zip(axes.flat, FEATURE_COLUMNS):
        values = frame[feature].to_numpy()
        plotted = np.sign(values) * np.log1p(np.abs(values))
        ax.hist(plotted, bins=35, color=colors[sensor], edgecolor='white')
        ax.set_title(feature, fontsize=9)
        ax.tick_params(labelsize=8)
    axes.flat[-1].set_visible(False)
    fig.suptitle(sensor + ': distribusi training pada sumbu sign(x) log(1 + |x|)', y=1.02)
    show_figure(fig, 'feature_distribution_' + sensor)
    correlation = frame[list(FEATURE_COLUMNS)].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    heat = ax.imshow(correlation, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(FEATURE_COLUMNS)), FEATURE_COLUMNS, rotation=70, ha='right', fontsize=8)
    ax.set_yticks(range(len(FEATURE_COLUMNS)), FEATURE_COLUMNS, fontsize=8)
    ax.set_title(sensor + ': korelasi Pearson fitur mentah training')
    fig.colorbar(heat, ax=ax, label='Pearson r', fraction=0.045)
    show_figure(fig, 'feature_correlation_' + sensor)

## 6. Scaling yang hanya belajar dari training

Setiap fitur dan sensor memiliki batas persentil tersendiri. Hanya baris IMF nyata milik complete-case training yang digunakan saat fit. Validation dan test memakai parameter yang sama. Nilai di luar batas fitur dipotong ke rentang 0 sampai 1; target tidak ditransformasi.

Jika rentang persentil nol, nilai fitur dipetakan ke titik tengah 0,5 dan ditandai dalam laporan. Ini mencegah pembagian nol dan tetap membedakan fitur nyata dari padding. Dataset aktual tidak memerlukan imputasi; nilai hilang/non-finite akan menghentikan pipeline.

In [ ]:
scalers = fit_scalers(tables, manifest)
display(pd.DataFrame([{'sensor': sensor, 'baris_fit_training': parameters['fit_rows'], 'pasien_fit_training': parameters['fit_patients'], 'fitur_rentang_persentil_nol': sum(parameters['zero_percentile_span'])} for sensor, parameters in scalers['sensors'].items()]))

## 7. Feature image, mask, dan penyimpanan

Masing-masing sensor disusun sebagai matriks 50 x 11. IMF1 sampai IMF50 menempati baris sesuai nomor, bukan urutan alfabet. IMF tidak tersedia diberi padding tanpa interpolasi. Fitur nyata dipetakan dari 0..1 ke piksel 1..255 dan dibulatkan ke uint8; padding memakai 0. Pembulatan memberi galat kuantisasi maksimum 0,5/254 pada skala 0..1. Versi fitur float32 dan mask boolean turut disimpan untuk audit dan baseline tabular.

Urutan channel adalah Flow, PulseRate, Pleth. Matriks gabungan 50 x 11 x 3 di-resize nearest-neighbor ke ukuran konfigurasi, awal 224 x 224 x 3. Tidak ada preprocessing backbone di sini. Sesuai migrasi PyTorch pada PHASE 2, model mengubah sumbu NHWC ke NCHW, membagi piksel dengan 255 tepat sekali, lalu memakai mean/std bobot TorchVision. Tidak ada crop atau resize tambahan.

Simpan NPZ per subset beserta manifest, parameter scaling, feature order, metadata, dan audit. Format NPZ memakai array numerik/string tanpa pickle.

In [ ]:
metadata, statistics, clipping = save_processed_dataset(tables, manifest, excluded, scalers, sources)
save_json(PROCESSED_DIR / 'data_audit_report.json', audit_report)
for sensor, frame in train_tables.items():
    frame[list(FEATURE_COLUMNS)].corr().to_csv(PROCESSED_DIR / ('training_correlation_' + sensor + '.csv'))
display(pd.DataFrame([{'subset': name.removesuffix('.npz'), 'image_shape': str(value['arrays']['images']['shape']), 'dtype': value['arrays']['images']['dtype'], 'ukuran_MiB': value['bytes'] / 1024**2} for name, value in metadata['files'].items()]).round(3))
clipping_summary = clipping.groupby(['split', 'sensor']).agg(fraksi_di_luar_batas_rata_rata=('outside_fraction', 'mean'), fitur_rentang_nol=('zero_percentile_span', 'sum')).reset_index()
display(clipping_summary.round(5))
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, sensor in zip(axes, SENSOR_FILES):
    selected = clipping[clipping.sensor == sensor].pivot(index='feature', columns='split', values='outside_fraction').reindex(FEATURE_COLUMNS)
    (100 * selected[['train', 'validation', 'test']]).plot.bar(ax=ax, color=['#2166ac', '#b35806', '#5e7d46'], width=0.8)
    ax.set(title=sensor, xlabel='', ylabel='Nilai fitur di luar batas (%)')
    ax.tick_params(axis='x', labelsize=7)
show_figure(fig, 'clipping_by_feature')

## 8. Visualisasi representasi

Gambar berikut merupakan feature image atau pseudo-image, bukan citra fisiologis asli. Warna gabungan menunjukkan pengkodean channel sensor, bukan warna jaringan atau organ. Kesamaan nomor IMF antar sensor tidak membuktikan kesamaan komponen frekuensi atau sinkronisasi fisiologis. Kegunaan representasi ini perlu diuji melalui eksperimen.

Contoh dipilih sebagai sampel training pertama secara deterministik, bukan berdasarkan target atau kualitas visual. Mask menunjukkan keberadaan IMF, bukan nilai fitur nol.

In [ ]:
with np.load(PROCESSED_DIR / 'train.npz', allow_pickle=False) as stored:
    feature_image = stored['feature_images'][0].copy()
    image = stored['images'][0].copy()
    mask = stored['imf_mask'][0].copy()
fig, axes = plt.subplots(1, 4, figsize=(14, 6))
for channel, sensor in enumerate(SENSOR_FILES):
    heat = axes[channel].imshow(feature_image[:, :, channel], cmap='gray', vmin=0, vmax=255, interpolation='nearest', aspect='auto')
    axes[channel].set(title=sensor, xlabel='Indeks fitur sesuai feature_order', ylabel='IMF')
    axes[channel].set_yticks([0, 9, 19, 29, 39, 49], [1, 10, 20, 30, 40, 50])
    fig.colorbar(heat, ax=axes[channel], fraction=0.04)
axes[3].imshow(feature_image, interpolation='nearest', aspect='auto')
axes[3].set(title='Gabungan 50 x 11 x 3', xlabel='Indeks fitur', ylabel='IMF')
axes[3].set_yticks([0, 9, 19, 29, 39, 49], [1, 10, 20, 30, 40, 50])
for ax in axes:
    ax.set_xticks([0, 5, 10], [1, 6, 11])
show_figure(fig, 'feature_image_channels')
fig, axes = plt.subplots(1, 2, figsize=(9, 5))
axes[0].imshow(mask, cmap='gray', vmin=0, vmax=1, interpolation='nearest', aspect='auto')
axes[0].set_xticks(range(len(SENSOR_FILES)), SENSOR_FILES, rotation=30)
axes[0].set(title='Mask IMF, putih berarti tersedia', ylabel='IMF')
axes[0].set_yticks([0, 9, 19, 29, 39, 49], [1, 10, 20, 30, 40, 50])
axes[1].imshow(image, interpolation='nearest')
axes[1].set(title='Hasil resize nearest-neighbor', xlabel='Kolom piksel', ylabel='Baris piksel')
show_figure(fig, 'feature_image_mask_resized')

## 9. Validasi akhir dan interpretasi aktual

Pemeriksaan mencakup tidak adanya overlap pasien/komponen/fingerprint, shape, finite values, mask, one-hot, target asli, dan kesamaan semua array setelah reload. Source hash diverifikasi kembali. Parameter preprocessing dan dataset yang tersimpan wajib dipakai seluruh notebook model tanpa splitting ulang.

Pemeriksaan ini membuktikan integritas pipeline, bukan performa prediksi atau validitas klinis. Salinan near-duplicate dan data sintetis yang tidak identik persis belum dapat dideteksi oleh aturan fingerprint ini.

In [ ]:
validate_manifest(manifest)
for sensor, source in sources.items():
    assert file_hash(DATA_DIR / SENSOR_FILES[sensor]) == source['sha256']
assert metadata['validation']['save_reload_equal']
assert metadata['validation']['patient_overlap'] == 0
assert metadata['validation']['component_overlap'] == 0
assert metadata['validation']['raw_fingerprint_overlap'] == 0
assert metadata['validation']['encoded_input_overlap'] == 0
save_json(PROCESSED_DIR / 'quality_checks.json', {'status': 'PASS', 'patient_overlap': 0, 'component_overlap': 0, 'raw_fingerprint_overlap': 0, 'encoded_input_overlap': 0, 'source_unchanged': True, 'save_reload_equal': True, 'finite_arrays': True, 'target_preserved': True, 'scaler_fit_subset': 'train', 'research_status': metadata['status']})
versions = {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'scipy', 'openpyxl', 'joblib', 'ipykernel', 'nbformat', 'nbclient']}
save_json(PROCESSED_DIR / 'environment_versions.json', {'python': sys.version, 'packages': versions, 'gpu_required': False})
summary = statistics[statistics.position == 'Overall']
split_text = '; '.join(f'{row.split}: {row.samples} sampel, {row.patients} ID, {row.components} kelompok' for row in summary.itertuples())
zero_span = sum(sum(item['zero_percentile_span']) for item in scalers['sensors'].values())
display(Markdown(f'**Hasil aktual:** {len(manifest):,} sampel dari {manifest.id.nunique()} ID membentuk {manifest.patient_component.nunique()} komponen. {split_text}. Sebanyak {len(excluded)} pasangan tidak lengkap dikeluarkan dari eksperimen utama. Maksimum AHI tetap {manifest.AHI.max():g}. Fitur dengan rentang persentil nol: {zero_span}. Semua pemeriksaan integritas dan reload lulus.'))
display(Markdown('**Batasan:** hasil preprocessing bersifat sementara sambil menunggu konfirmasi klien. Grouping hanya mengurangi risiko kebocoran dari salinan persis; tidak memastikan provenance atau kebenaran label. Belum ada model dilatih. Tahap berikutnya adalah smoke test ResNet50 setelah persetujuan PHASE 2 dan setup GPU.'))